In [26]:
import xarray as xr
import numpy as np
import pandas as pd
from libpysal.weights import lat2W
from esda import Moran
np.random.seed(12345)
import warnings
warnings.filterwarnings("ignore", message=".*is an island.*")


In [19]:
def moran_timeseries(da, years, lat_dim="lat", lon_dim="lon"):
    """计算每一年的 Moran's I 指数"""
    nlat = da.sizes[lat_dim]
    nlon = da.sizes[lon_dim]

    w = lat2W(nlat, nlon)
    w.transform = "r"

    moran_list = []

    for i, y in enumerate(years):
        arr = da.isel(time=i).values
        arr = np.nan_to_num(arr, nan=0.0)

        x = arr.flatten()

        mi = Moran(x, w,permutations=0)
        moran_list.append(mi.I)

    return moran_list


In [20]:
years = list(range(2010, 2101, 10))

ds = xr.open_dataset(f"../../../NC/compare_diff.nc")
region_diff = ds["region_agri_diff"]
basin_diff = ds["basin_agri_diff"]

region_moran = moran_timeseries(region_diff, years)
basin_moran  = moran_timeseries(basin_diff, years)

df_global = pd.DataFrame({
    "year": years,
    "region_moran": region_moran,
    "basin_moran": basin_moran,
})
df_global_long = df_global.melt(
    id_vars="year",
    value_vars=["region_moran", "basin_moran"],
    var_name="mode_tmp",
    value_name="moran_I"
)

df_global_long["mode"] = df_global_long["mode_tmp"].str.replace("_moran", "")
df_global_long["scope"] = "global"
df_global_long["region"] = "ALL"

df_global_long = df_global_long[["year", "scope", "region", "mode", "moran_I"]]

In [21]:
import xarray as xr
import pandas as pd
import numpy as np

from libpysal.weights import lat2W
from libpysal.weights.util import w_subset
from esda import Moran

np.random.seed(12345)

In [22]:
def load_region_matrix(csv_path, lat_size=360, lon_size=720):
    df = pd.read_csv(csv_path)

    region_matrix = np.empty((lat_size, lon_size), dtype=object)
    region_matrix[:] = None

    for _, r in df.iterrows():
        I = int(r["I"]) - 1
        J = int(r["J"]) - 1
        region_matrix[I, J] = r["Rall"]

    return region_matrix

In [23]:
def regional_moran_timeseries(
    da,
    years,
    region_matrix,
    regions,
    w_full,
):
    """
    da: xarray.DataArray (time, lat, lon)
    returns: list of dict rows
    """

    results = []

    region_flat = region_matrix.flatten()

    for ti, year in enumerate(years):
        print(f"Processing year {year}")

        arr = da.isel(time=ti).values
        arr = np.nan_to_num(arr, nan=0.0)
        x_full = arr.flatten()

        for reg in regions:
            mask = region_flat == reg

            # 若该 region 没有格点（安全检查）
            if mask.sum() < 10:
                continue

            x_reg = x_full[mask]

            # 子权重矩阵（PySAL 官方推荐方式）
            w_reg = w_subset(w_full, np.where(mask)[0])
            w_reg.transform = "r"

            mi = Moran(x_reg, w_reg, permutations=0)

            results.append({
                "year": year,
                "region": reg,
                "moran_I": mi.I
            })

    return results


In [28]:
CSV_PATH = "../../../CSV/gridset/RIJ_17regions.csv"
NC_PATH  = "../../../NC/compare_diff.nc"
OUT_CSV  = "../../../CSV/moran_interannual_diff_by_region.csv"

years = list(range(2010, 2101, 10))

region_matrix = load_region_matrix(CSV_PATH, 360, 720)

regions = sorted({
    r for r in region_matrix.flatten()
    if isinstance(r, str) and r.strip() != ""
})

print("Regions:", regions)

# --- 构造全局空间权重 ---
nlat, nlon = 360, 720
w_full = lat2W(nlat, nlon)
w_full.transform = "r"

# --- 读取 NetCDF ---
ds = xr.open_dataset(NC_PATH)

results_all = []

# --- region aggregation ---
print("\n=== REGION MODE ===")
res_region = regional_moran_timeseries(
    ds["region_agri_diff"],
    years,
    region_matrix,
    regions,
    w_full
)
for r in res_region:
    r["mode"] = "region"
results_all.extend(res_region)

# --- basin aggregation ---
print("\n=== BASIN MODE ===")
res_basin = regional_moran_timeseries(
    ds["basin_agri_diff"],
    years,
    region_matrix,
    regions,
    w_full
)
for r in res_basin:
    r["mode"] = "basin"
results_all.extend(res_basin)

# --- 输出 CSV ---
df_regional = pd.DataFrame(results_all)
df_regional["scope"] = "regional"
df_regional = df_regional[["year", "scope", "region", "mode", "moran_I"]]
df_all = pd.concat(
    [df_global_long, df_regional],
    ignore_index=True
)
df_all["moran_I"] = df_all["moran_I"].round(3)
df_all.to_csv(
    "../../../CSV/moran_interannual_diff_global_and_regional.csv",
    index=False
)

print("[Saved] moran_interannual_diff_global_and_regional.csv")


Regions: ['BRA', 'CAN', 'CHN', 'CIS', 'IND', 'JPN', 'TUR', 'USA', 'XAF', 'XE25', 'XER', 'XLM', 'XME', 'XNF', 'XOC', 'XSA', 'XSE']

=== REGION MODE ===
Processing year 2010
('WARNING: ', 128461, ' is an island (no neighbors)')
('WARNING: ', 134935, ' is an island (no neighbors)')
('WARNING: ', 47003, ' is an island (no neighbors)')
('WARNING: ', 51317, ' is an island (no neighbors)')
('WARNING: ', 68595, ' is an island (no neighbors)')
('WARNING: ', 97803, ' is an island (no neighbors)')
('WARNING: ', 14113, ' is an island (no neighbors)')
('WARNING: ', 17107, ' is an island (no neighbors)')
('WARNING: ', 18673, ' is an island (no neighbors)')
('WARNING: ', 20105, ' is an island (no neighbors)')
('WARNING: ', 26648, ' is an island (no neighbors)')
('WARNING: ', 34581, ' is an island (no neighbors)')
('WARNING: ', 59689, ' is an island (no neighbors)')
('WARNING: ', 69021, ' is an island (no neighbors)')
('WARNING: ', 116465, ' is an island (no neighbors)')
('WARNING: ', 81999, ' is an i

f:\anaconda\Lib\site-packages\libpysal\weights\set_operations.py:386: UserWarning: The weights matrix is not fully connected: 
 There are 3 disconnected components.
 There are 2 islands with ids: 128461, 134935.
  return W(neighbors, id_order=list(ids), **kwargs)
f:\anaconda\Lib\site-packages\libpysal\weights\set_operations.py:386: UserWarning: The weights matrix is not fully connected: 
 There are 14 disconnected components.
 There are 3 islands with ids: 47003, 51317, 68595.
  return W(neighbors, id_order=list(ids), **kwargs)
f:\anaconda\Lib\site-packages\libpysal\weights\set_operations.py:386: UserWarning: The weights matrix is not fully connected: 
 There are 3 disconnected components.
 There is 1 island with id: 97803.
  return W(neighbors, id_order=list(ids), **kwargs)
f:\anaconda\Lib\site-packages\libpysal\weights\set_operations.py:386: UserWarning: The weights matrix is not fully connected: 
 There are 34 disconnected components.
 There are 8 islands with ids: 14113, 17107, 186

('WARNING: ', 127094, ' is an island (no neighbors)')
('WARNING: ', 129253, ' is an island (no neighbors)')
('WARNING: ', 143012, ' is an island (no neighbors)')
('WARNING: ', 158166, ' is an island (no neighbors)')
('WARNING: ', 196995, ' is an island (no neighbors)')
('WARNING: ', 78145, ' is an island (no neighbors)')
('WARNING: ', 81687, ' is an island (no neighbors)')
('WARNING: ', 88164, ' is an island (no neighbors)')
('WARNING: ', 111823, ' is an island (no neighbors)')
('WARNING: ', 148050, ' is an island (no neighbors)')
('WARNING: ', 15540, ' is an island (no neighbors)')
('WARNING: ', 66614, ' is an island (no neighbors)')
('WARNING: ', 66624, ' is an island (no neighbors)')
('WARNING: ', 207726, ' is an island (no neighbors)')
('WARNING: ', 83030, ' is an island (no neighbors)')
('WARNING: ', 92361, ' is an island (no neighbors)')
('WARNING: ', 94531, ' is an island (no neighbors)')
('WARNING: ', 97408, ' is an island (no neighbors)')
('WARNING: ', 98066, ' is an island (n

f:\anaconda\Lib\site-packages\libpysal\weights\set_operations.py:386: UserWarning: The weights matrix is not fully connected: 
 There are 10 disconnected components.
 There are 3 islands with ids: 159710, 171364, 201597.
  return W(neighbors, id_order=list(ids), **kwargs)
f:\anaconda\Lib\site-packages\libpysal\weights\set_operations.py:386: UserWarning: The weights matrix is not fully connected: 
 There are 92 disconnected components.
 There are 52 islands with ids: 106491, 108698, 109370, 113733, 115836, 117341, 119463, 119497, 120026, 120196, 121659, 123097, 123625, 123627, 123881, 125182, 127226, 129587, 130293, 131028, 131750, 131752, 132405, 133216, 134589, 135377, 135410, 140478, 141838, 144664, 145385, 145496, 147594, 149781, 150549, 151928, 151983, 152696, 152715, 154157, 154803, 154878, 156240, 156959, 157037, 157041, 157689, 157691, 159099, 159929, 160638, 161269.
  return W(neighbors, id_order=list(ids), **kwargs)
f:\anaconda\Lib\site-packages\libpysal\weights\set_operations

('WARNING: ', 159710, ' is an island (no neighbors)')
('WARNING: ', 171364, ' is an island (no neighbors)')
('WARNING: ', 201597, ' is an island (no neighbors)')
('WARNING: ', 106491, ' is an island (no neighbors)')
('WARNING: ', 108698, ' is an island (no neighbors)')
('WARNING: ', 109370, ' is an island (no neighbors)')
('WARNING: ', 113733, ' is an island (no neighbors)')
('WARNING: ', 115836, ' is an island (no neighbors)')
('WARNING: ', 117341, ' is an island (no neighbors)')
('WARNING: ', 119463, ' is an island (no neighbors)')
('WARNING: ', 119497, ' is an island (no neighbors)')
('WARNING: ', 120026, ' is an island (no neighbors)')
('WARNING: ', 120196, ' is an island (no neighbors)')
('WARNING: ', 121659, ' is an island (no neighbors)')
('WARNING: ', 123097, ' is an island (no neighbors)')
('WARNING: ', 123625, ' is an island (no neighbors)')
('WARNING: ', 123627, ' is an island (no neighbors)')
('WARNING: ', 123881, ' is an island (no neighbors)')
('WARNING: ', 125182, ' is a